# Example Usage

This notebook demonstrates the use of a dataset class defined outside of Hyrax that is used to train 1) the HyraxCNN model
and 2) the local VGG11 model.

We'll download the Galaxy10 dataset from AstroNN to use in this example. It's a collection of ~23k labeled galaxy images from 10 possible classes.

More info here: https://github.com/henrysky/astroNN and here: https://astronn.readthedocs.io/en/stable/galaxy10sdss.html

In [1]:
import pooch

file_path = pooch.retrieve(
    # DOI for Example HSC dataset
    url="http://www.astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5",
    known_hash="sha256:969A6B1CEFCC36E09FFFA86FEBD2F699A4AA19B837BA0427F01B0BC6DED458AF",
    fname="Galaxy10.h5",
    path="./data/galaxy_10",
)

In [ ]:
from hyrax import Hyrax

h = Hyrax()

We'll specify that we want to use the local Galaxy10 dataset class.

In [ ]:
data_request = {
    "train": {
        "data": {
            "dataset_class": "external_hyrax_example.datasets.galaxy10_dataset.Galaxy10Dataset",
            "data_location": "/home/drew/code/external_hyrax_example/docs/pre_executed/data/galaxy_10",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
            "dataset_config": {
                "external_hyrax_example": {
                    "galaxy10_dataset": {
                        "channels_first": True,
                    },
                },
            },
        },
    },
    "validate": {
        "data": {
            "dataset_class": "external_hyrax_example.datasets.galaxy10_dataset.Galaxy10Dataset",
            "data_location": "/home/drew/code/external_hyrax_example/docs/pre_executed/data/galaxy_10",
            "fields": ["image", "label"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
            "dataset_config": {
                "external_hyrax_example": {
                    "galaxy10_dataset": {
                        "channels_first": True,
                    },
                },
            },
        },
    },
}

h.set_config("data_request", data_request)

We'll start by training the HyraxCNN model. This model is from the Hyrax package.

In [ ]:
h.set_config("model.name", "HyraxCNN")

In [ ]:
ds = h.prepare()

We'll need to update the prepare_inputs function that HyraxCNN uses because the dimensions of the Galaxy10 images
are in a different order than what HyraxCNN expects.

In [ ]:
@staticmethod
def prepare_inputs(data_dict):
    import numpy as np

    if "data" not in data_dict:
        raise RuntimeError("Unable to find `data` key in data_dict")

    data = data_dict["data"]
    image = np.asarray(data["image"], dtype=np.float32)

    # normalize the image to have mean 0.5 and std 0.5
    image = np.asarray(image, dtype=np.float32)
    mean = np.asarray([0.5, 0.5, 0.5], dtype=np.float32)
    std = np.asarray([0.5, 0.5, 0.5], dtype=np.float32)

    mean = mean[:, None, None]
    std = std[:, None, None]

    normalized_image = (image - mean) / (std + 1e-8)

    label = None
    if "label" in data:
        label = np.asarray(data["label"], dtype=np.int64)

    return (normalized_image, label)

In [ ]:
model = h.model()
model.prepare_inputs = prepare_inputs

Train the model

In [ ]:
model = h.train()

Now we'll update the model that we're training to use the locally defined VGG11 model.
We'll go ahead and update the `prepare_inputs` function for this model as well to appropriately handle
the dimensions of the Galaxy10 data.

In [ ]:
h.set_config("model.name", "external_hyrax_example.models.vgg11.VGG11")

In [ ]:
model = h.model()
model.prepare_inputs = prepare_inputs

Now we'll train the VGG11 model.

In [ ]:
model = h.train()

Quick loss comparison between the models.

Orange = HyraxCNN
Green = VGG11


![loss_values](../_static/HyraxCNN_vs_VGG11.png)